# FIREGUARD ID v2 — Minggu 4: [M -> O] Desain Deep Learning (Deep Embedded Clustering / DEC)
**Sesuai Ketentuan Minggu 4 (Bab 6.3.1–6.3.3 & 6.3.5):** Pemilihan arsitektur AE simetris (`4->32->16->4`), jumlah parameter, preprocessing khusus, dan verifikasi training awal (loss menurun).

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import json
import numpy as np
import pandas as pd
from src.utils import load_yaml, resolve_path

proj_cfg = load_yaml("config/project.yaml", repo_root=REPO_ROOT)
print("Repo root:", REPO_ROOT)

Repo root: FIREGUARD-ID


In [2]:
import torch
from src.models.dec_model import SymmetricAutoencoder, DECModel, DECTrainConfig, train_dec_single_run
dec_exp = json.loads(resolve_path("reports/clustering/dec_experiments.json", REPO_ROOT).read_text(encoding="utf-8"))
print("Architecture & Param Counts:", json.dumps(dec_exp["architecture"], indent=2))

# Sanity check initial forward + short 10-epoch run verifying MSE/KL decrease
df_mat = pd.read_parquet(resolve_path("data/processed/clustering_matrix_scaled.parquet", REPO_ROOT))
X = df_mat[[c for c in df_mat.columns if c.endswith("_scaled")]].to_numpy(dtype=np.float32)
sanity_cfg = DECTrainConfig(n_clusters=3, latent_dim=4, hidden_dims=[32, 16], pretrain_epochs=15, pretrain_lr=1e-3, finetune_max_iter=20, finetune_lr=5e-4, batch_size=64, update_interval=10, tol=1e-4, alpha=1.0, gamma_recon=0.1, seed=42)
sanity = train_dec_single_run(X, sanity_cfg, torch.device("cpu"))
print("Sanity pretrain MSE:", sanity["pretrain"]["initial_mse"], "->", sanity["pretrain"]["final_mse"])
print("Sanity finetune epochs:", sanity["finetune"]["epochs_run"], "| KL:", sanity["finetune"]["final_kl_loss"])

Architecture & Param Counts: {
  "encoder": [
    4,
    32,
    16,
    4
  ],
  "decoder": [
    4,
    16,
    32,
    4
  ],
  "hidden_activation": "ReLU",
  "latent_activation": "Linear",
  "output_activation": "Linear",
  "student_t_alpha": 1.0,
  "gamma_recon": 0.1,
  "total_params": 1524,
  "trainable_params": 1524,
  "encoder_params": 756,
  "decoder_params": 756,
  "cluster_centers_params": 12
}


C:\Users\ghofu\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\ghofu\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\ghofu\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\ghofu\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\ghofu\AppData\Local\Programs\Python\Python310\lib\subp

Sanity pretrain MSE: 1.057939 -> 0.035905
Sanity finetune epochs: 20 | KL: 0.075775
